# H&M Personalized Fashion Recommendations

## Overview KPI Analysis

### 1. Total Revenue

In [1]:
SELECT MIN(t_dat) as from_date, MAX(t_dat) as until_date, 
    FORMAT_NUMBER(SUM(price) * 1000, 2) AS total_revenue
FROM transactions;

StatementMeta(, f875734d-590d-4d0d-8d4e-89c37f1e7e7e, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

### 2. Total Transaction 

In [2]:
SELECT MIN(t_dat) as from_date, MAX(t_dat) as until_date, 
    count(*) as total_transactions
FROM transactions;

StatementMeta(, 682a41c4-e2c9-4b71-a2f5-f0c1eb15fd14, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

### 3. Total Customers

In [3]:
SELECT MIN(t_dat) as from_date, MAX(t_dat) as until_date, 
    COUNT( DISTINCT customer_id) as total_transactions
FROM transactions;

StatementMeta(, 682a41c4-e2c9-4b71-a2f5-f0c1eb15fd14, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

### 4. 	Average Order Value (AOV)

In [3]:
SELECT ROUND(AVG(price * 1000), 2) AS average_price
FROM transactions;

StatementMeta(, 223ec93d-edcd-446e-9606-51ecbd26137d, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

### 5. Sales between Black Friday and Other Non Black Friday Periods

In [4]:
WITH YearlyRevenue AS (
    SELECT
        YEAR(t_dat) AS sales_year,
        MONTH(t_dat) AS sales_month,
        SUM(price) AS monthly_revenue
    FROM dbo.transactions
    GROUP BY YEAR(t_dat), MONTH(t_dat)
)
SELECT
    sales_year,

    CAST(SUM(CASE
        WHEN sales_month = 11  THEN monthly_revenue
        ELSE 0
    END) AS DECIMAL(18, 2)) AS november_revenue,

    CAST(
        SUM(CASE
            WHEN sales_month <> 11 THEN monthly_revenue
            ELSE 0
        END) / 11.0
        AS DECIMAL(18, 2)
    ) AS average_other_monthly_revenue,

    CAST(
        (
            SUM(CASE WHEN sales_month = 11 THEN monthly_revenue ELSE 0 END)
            /
            NULLIF(
                SUM(CASE WHEN sales_month <> 11 THEN monthly_revenue ELSE 0 END) / 11.0,
                0
            ) - 1
        ) * 100
        AS DECIMAL(10, 2)
    ) AS november_revenue_difference_pct

FROM YearlyRevenue
WHERE sales_year < 2020
GROUP BY sales_year
ORDER BY sales_year;

StatementMeta(, 00699708-9e1c-42fc-b6a0-613165a55323, 5, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 4 fields>

## Product Performance Analysis


### 1. Top 10 Department By Revenue

In [10]:
SELECT a.department_name, FORMAT_NUMBER(SUM(t.price) * 1000, 2) AS Total_revenue
FROM transactions t
LEFT JOIN articles a
ON t.article_id = a.article_id
GROUP BY a.department_name
ORDER BY SUM(t.price) desc
LIMIT 10;


StatementMeta(, f875734d-590d-4d0d-8d4e-89c37f1e7e7e, 11, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

### 2. Which product categories generate the most revenue?

In [21]:
SELECT a.product_group_name AS Prod_categories,
    FORMAT_NUMBER(SUM(t.price) * 1000, 2) AS Total_revenue
FROM transactions t
LEFT JOIN articles a
    ON a.article_id = t.article_id
GROUP BY 1
ORDER BY SUM(t.price) DESC
LIMIT 10;

StatementMeta(, e4b81289-b613-42ff-acb9-8281b61d6c76, 21, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

### 3. Which Products generate the most revenue?

In [2]:
SELECT a.product_type_name AS products,
    FORMAT_NUMBER(SUM(t.price) * 1000, 2) AS Total_revenue
FROM transactions t
LEFT JOIN articles a
    ON a.article_id = t.article_id
GROUP BY 1
ORDER BY SUM(t.price) DESC
LIMIT 10;

StatementMeta(, d61ac27d-2ca8-48bc-bf8f-5b3fcdf0ffa7, 3, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

## Customer Behavior Overview

### 1. How many active customers?

In [6]:
SELECT COUNT(DISTINCT customer_id) as active_customers
FROM customers
WHERE Active =1;

StatementMeta(, 5c407cc6-09e5-4788-92c0-4aee77989450, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

### 2. Overall average purchase frequency

In [12]:
WITH customer_frequency AS (
    SELECT
        customer_id,
        COUNT(DISTINCT t_dat) AS purchase_days
    FROM transactions
    GROUP BY customer_id
)
SELECT
    ROUND(AVG(purchase_days * 1.0),1) AS avg_purchase_days_per_customer
FROM customer_frequency;

StatementMeta(, 5c407cc6-09e5-4788-92c0-4aee77989450, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

### 3. Average Customer Age

In [17]:
SELECT ROUND(AVG(age),2) as average_age
FROM transactions t
LEFT JOIN customers c
ON c.customer_id = t.customer_id;

StatementMeta(, cc92e010-9ae8-4112-b8a6-82a1a91aebeb, 18, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

### 4. Which Month Did Customers Buy The Most?

In [12]:
SELECT
    MONTH(t_dat) AS month,
    COUNT(*) AS total_items_purchased,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(
        COUNT(*) * 1.0 / COUNT(DISTINCT customer_id), 2
    ) AS items_per_customer
FROM transactions
GROUP BY MONTH(t_dat)
ORDER BY total_items_purchased DESC;

StatementMeta(, a40e9eb9-6677-4d1e-a9e9-e751634eb965, 13, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 4 fields>

### 5. Which weekdays have the highest purchasing activity?

In [17]:
SELECT
    date_format(t_dat, 'EEEE') AS day_name,
    COUNT(*) AS total_items_purchased,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM transactions
GROUP BY date_format(t_dat, 'EEEE'), dayofweek(t_dat)
ORDER BY 2 DESC;

StatementMeta(, a40e9eb9-6677-4d1e-a9e9-e751634eb965, 18, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 3 fields>

### 6. Customer-Level Article Purchase Frequency and Repeat Purchase Days

In [8]:
WITH items_bought_per_day AS (
    SELECT
        t_dat,
        customer_id,
        article_id,
        COUNT(*) AS total_items_purchased
    FROM transactions
    GROUP BY
        t_dat,
        customer_id,
        article_id
)

SELECT
    customer_id,
    article_id,
    SUM(total_items_purchased) AS total_bought,
    COUNT(DISTINCT t_dat) AS purchase_days
FROM items_bought_per_day
GROUP BY
    customer_id,
    article_id
ORDER BY purchase_days DESC;

StatementMeta(, d4da8b2d-b9b1-4d27-8151-495439417282, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 4 fields>

### 7. Which product types are most popular among customers recently?

In [11]:
WITH date_limit AS (
    SELECT
        MAX(t_dat) AS max_date,
        DATE_SUB(MAX(t_dat), 6) AS start_date
    FROM transactions
)

SELECT
    a.product_type_name,
    COUNT(*) AS total_items_purchased,
    COUNT(DISTINCT t.customer_id) AS unique_customers,
    ROUND(SUM(t.price) * 1000, 2) AS total_spending
FROM transactions t
LEFT JOIN articles a
    ON t.article_id = a.article_id
CROSS JOIN date_limit d
WHERE t.t_dat BETWEEN d.start_date AND d.max_date
GROUP BY
    a.product_type_name
ORDER BY
    total_items_purchased DESC;

StatementMeta(, 1b0849aa-2d5e-495e-90ed-effbbf340c12, 5, Finished, Available, Finished, False)

<Spark SQL result set with 113 rows and 4 fields>

## Customer Behavior

### 1. Which product types and colours are customers currently interested in?

In [16]:
WITH date_limit AS (
    SELECT MAX(t_dat) AS max_date
    FROM transactions
)

SELECT
    t.customer_id,
    a.product_type_name, 
    a.colour_group_name,    
    FORMAT_NUMBER(SUM(t.price) * 1000, 2) AS total_spending,
    COUNT(*) AS total_purchased,
    DATEDIFF(d.max_date, t.t_dat) AS days_before_latest_transaction

FROM transactions t
LEFT JOIN articles a
    ON t.article_id = a.article_id
CROSS JOIN date_limit d
GROUP BY 1, 2, 3, 6
ORDER BY total_purchased DESC;

StatementMeta(, 7c1f0c33-cb60-4b94-b644-5374973baf7e, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 6 fields>

### 2. Which specific product are customers currently interested in?

In [ ]:
WITH date_limit AS (
    SELECT MAX(t_dat) AS max_date
    FROM transactions
)

SELECT
    t.customer_id,
    t.article_id,
    a.prod_name,
    COUNT(*) AS purchase_frequency,
    ROUND(SUM(t.price) * 1000, 2) AS total_spending,
    ROUND(AVG(t.price) * 1000, 2) AS average_item_price,
    DATEDIFF(d.max_date, MAX(t.t_dat)) AS days_since_last_purchase

FROM transactions t

LEFT JOIN articles a
    ON t.article_id = a.article_id

CROSS JOIN date_limit d

GROUP BY
    t.customer_id,
    t.article_id,
    a.prod_name,
    d.max_date

ORDER BY
    t.customer_id,
    purchase_frequency DESC;

StatementMeta(, 7c1f0c33-cb60-4b94-b644-5374973baf7e, 13, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 7 fields>

### 3. Which products are frequently purchased together?

In [ ]:
WITH baskets AS (
    SELECT DISTINCT
        customer_id,
        t_dat,
        article_id
    FROM transactions
),

product_pairs AS (
    SELECT
        a.article_id AS article_1,
        b.article_id AS article_2,
        COUNT(*) AS times_purchased_together,
        COUNT(DISTINCT a.customer_id) AS unique_customers
    FROM baskets a
    JOIN baskets b
        ON a.customer_id = b.customer_id
        AND a.t_dat = b.t_dat
        AND a.article_id < b.article_id
    GROUP BY
        a.article_id,
        b.article_id
)

SELECT
    p.article_1,
    a1.prod_name AS product_1,
    a1.product_type_name AS product_type_1,

    p.article_2,
    a2.prod_name AS product_2,
    a2.product_type_name AS product_type_2,

    p.times_purchased_together,
    p.unique_customers

FROM product_pairs p
LEFT JOIN articles a1
    ON p.article_1 = a1.article_id
LEFT JOIN articles a2
    ON p.article_2 = a2.article_id

WHERE p.times_purchased_together >= 5

ORDER BY
    p.times_purchased_together DESC;

StatementMeta(, 7c1f0c33-cb60-4b94-b644-5374973baf7e, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 8 fields>

### 4. Customer Recommendation

#### 4.1 Creating customer_preferences

In [ ]:
CREATE OR REPLACE TABLE customer_preferences
USING DELTA
AS

SELECT
    customer_id,
    article_id,
    COUNT(*) AS purchase_frequency,
    ROUND(SUM(price) * 1000, 2) AS total_spending,
    MAX(t_dat) AS last_purchase_date

FROM transactions

WHERE article_id IS NOT NULL

GROUP BY
    customer_id,
    article_id;

StatementMeta(, 7c1f0c33-cb60-4b94-b644-5374973baf7e, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### 4.2 Creating product_pairs Table

In [ ]:
CREATE OR REPLACE TABLE product_pairs
USING DELTA
AS

WITH baskets AS (
    SELECT DISTINCT
        customer_id,
        t_dat,
        article_id
    FROM transactions
    WHERE article_id IS NOT NULL
),

pairs AS (
    SELECT
        a.article_id AS article_1,
        b.article_id AS article_2,
        COUNT(*) AS times_purchased_together,
        COUNT(DISTINCT a.customer_id) AS unique_customers
    FROM baskets a
    INNER JOIN baskets b
        ON a.customer_id = b.customer_id
        AND a.t_dat = b.t_dat
        AND a.article_id < b.article_id
    GROUP BY
        a.article_id,
        b.article_id
)

SELECT *
FROM pairs
WHERE times_purchased_together >= 5;

StatementMeta(, 7c1f0c33-cb60-4b94-b644-5374973baf7e, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### 4.3 Resutls

In [ ]:
CREATE OR REPLACE TABLE customer_recommendations
USING DELTA
AS

WITH directional_pairs AS (

    -- First recommendation direction
    SELECT
        article_1 AS purchased_article,
        article_2 AS recommended_article,
        times_purchased_together
    FROM product_pairs

    UNION ALL

    -- Reverse recommendation direction
    SELECT
        article_2 AS purchased_article,
        article_1 AS recommended_article,
        times_purchased_together
    FROM product_pairs
),

recommendations AS (

    -- Combine customer preferences with product associations
    SELECT
        cp.customer_id,
        p.recommended_article,

        SUM(
            cp.purchase_frequency *
            p.times_purchased_together
        ) AS recommendation_score

    FROM customer_preferences cp

    INNER JOIN directional_pairs p
        ON cp.article_id = p.purchased_article

    GROUP BY
        cp.customer_id,
        p.recommended_article
),

new_recommendations AS (

    -- Exclude articles already purchased
    SELECT r.*
    FROM recommendations r

    LEFT ANTI JOIN customer_preferences cp
        ON r.customer_id = cp.customer_id
        AND r.recommended_article = cp.article_id
),

ranked_recommendations AS (

    -- Rank recommendations for each customer
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY recommendation_score DESC
        ) AS recommendation_rank

    FROM new_recommendations
)

SELECT
    r.customer_id,
    r.recommended_article AS article_id,
    a.prod_name,
    a.product_type_name,
    a.colour_group_name,
    r.recommendation_score,
    r.recommendation_rank

FROM ranked_recommendations r

LEFT JOIN articles a
    ON r.recommended_article = a.article_id

WHERE r.recommendation_rank <= 5;

StatementMeta(, 7c1f0c33-cb60-4b94-b644-5374973baf7e, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
SELECT * FROM customer_recommendations
LIMIT 1000;

StatementMeta(, 00699708-9e1c-42fc-b6a0-613165a55323, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 7 fields>